In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Lodhi Road, Delhi - IMD.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,BP,Xylene,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,139.04,279.78,47.11,18.17,43.83,0.73,10.51,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,133.16,267.82,32.95,23.33,36.74,0.76,11.30,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,178.87,355.92,92.18,27.95,78.48,1.24,12.42,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,211.35,367.58,89.47,74.74,89.12,1.50,18.10,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,132.16,265.13,26.04,58.56,52.32,0.74,10.13,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,282.70,416.83,48.53,31.08,55.99,1.40,37.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,245.65,379.46,42.49,34.39,52.84,1.38,34.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,192.29,307.81,33.80,36.53,46.91,1.30,40.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,203.54,321.81,42.47,35.16,53.23,1.33,45.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 12)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene', 'Eth-Benzene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:



# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 10)
          From Date           To Date  PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  58.89  279.78  47.11  18.17  43.83   
1  02-01-2025 00:00  03-01-2025 00:00  58.89  267.82  32.95  23.33  36.74   
2  03-01-2025 00:00  04-01-2025 00:00  58.89  157.08  10.90  27.95  78.48   
3  04-01-2025 00:00  05-01-2025 00:00  58.89  157.08  10.90  22.21  20.74   
4  05-01-2025 00:00  06-01-2025 00:00  58.89  265.13  26.04  22.21  52.32   

     CO  Ozone  TOT-RF  
0  0.73  10.51       0  
1  0.76  11.30       0  
2  1.24  12.42       0  
3  0.66  18.10       0  
4  0.74  10.13       0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,0.073631,2.279362,2.342270,-0.621141,1.255436,0.441841,-1.455624,0.0
1,02-01-2025 00:00,03-01-2025 00:00,0.073631,2.063815,1.307694,0.414084,0.811878,0.587603,-1.416702,0.0
2,03-01-2025 00:00,04-01-2025 00:00,0.073631,0.068018,-0.303352,1.340971,3.423181,2.919797,-1.361521,0.0
3,04-01-2025 00:00,05-01-2025 00:00,0.073631,0.068018,-0.303352,0.189384,-0.189101,0.101730,-1.081676,0.0
4,05-01-2025 00:00,06-01-2025 00:00,0.073631,2.015335,0.802826,0.189384,1.786581,0.490429,-1.474346,0.0
...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,0.073631,0.068018,2.446020,1.968927,2.016180,0.101730,-0.121927,0.0
316,13-11-2025 00:00,14-11-2025 00:00,0.073631,0.068018,2.004718,2.632995,1.819112,0.101730,-0.284513,0.0
317,14-11-2025 00:00,15-11-2025 00:00,0.073631,0.068018,1.369798,3.062333,1.448125,0.101730,0.018981,0.0
318,15-11-2025 00:00,16-11-2025 00:00,0.073631,0.068018,2.003256,2.787476,1.843511,0.101730,0.276162,0.0


In [10]:
df.to_excel('lodhiroadIMD2025.xlsx', index=False)